# Milestone 3 – ResNet-18 on CIFAR-10 (With Code Smells)

Pretrained ResNet-18 with **4 injected ML code smells**:
1. Unbounded Graph Expansion
2. Graph-Constant Bottleneck
3. GPU Released Memory Failure
4. Shape Mismatch Leak

In [ ]:
!pip install -q codecarbon scipy

In [ ]:
import numpy as np
import pandas as pd
import gc, os, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models

from sklearn.metrics import classification_report
from codecarbon import EmissionsTracker


In [ ]:
import torchvision.datasets as dsets
import torchvision.transforms as transforms

print("Loading CIFAR-10 dataset...")
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2023, 0.1994, 0.2010)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

train_set = dsets.CIFAR10(root='/tmp/cifar10', train=True,  download=True, transform=transform)
test_set  = dsets.CIFAR10(root='/tmp/cifar10', train=False, download=True, transform=transform)

# Pre-extract to numpy for consistent handling
X_train_np = np.stack([train_set[i][0].numpy() for i in range(len(train_set))]).astype(np.float32)
y_train_np = np.array([train_set[i][1] for i in range(len(train_set))], dtype=np.int64)
X_test_np  = np.stack([test_set[i][0].numpy()  for i in range(len(test_set))]).astype(np.float32)
y_test_np  = np.array([test_set[i][1] for i in range(len(test_set))], dtype=np.int64)
print(f"X_train: {X_train_np.shape}  X_test: {X_test_np.shape}")


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

def build_resnet18(num_classes=10):
    """Pretrained ResNet-18 with replaced final layer."""
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(y_batch).sum().item()
        total += X_batch.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(y_batch).sum().item()
            total += X_batch.size(0)
    return total_loss / total, correct / total

def get_precision_recall(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            outputs = model(X_batch)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())
    from sklearn.metrics import precision_score, recall_score
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels,  all_preds, average='macro', zero_division=0)
    return prec, rec


## Training – 10 Runs (Smelly)

In [ ]:
# ============================================================
# CODE SMELLS ARE INTENTIONALLY INJECTED FOR RESEARCH PURPOSES
# ============================================================
os.makedirs("emissions", exist_ok=True)
results = []

BATCH_SIZE = 32
EPOCHS     = 30

# SMELL 4 – Shape Mismatch Leak
# Tensors are unnecessarily upcast to float64 (doubles memory),
# then reshaped to an intermediate 5-D layout before being cast
# back. Both the float64 array and the reshaped intermediate
# remain allocated simultaneously.
X_train_f64  = X_train_np.astype(np.float64)          # 2× memory footprint
X_test_f64   = X_test_np.astype(np.float64)
N_tr, C, H, W = X_train_f64.shape
X_train_5d   = X_train_f64.reshape(N_tr, C, H, W, 1)  # unnecessary 5-D copy
X_train_back = X_train_5d.reshape(N_tr, C, H, W)       # reshape back; 5-D still in memory
N_te         = X_test_f64.shape[0]
X_test_5d    = X_test_f64.reshape(N_te, C, H, W, 1)
X_test_back  = X_test_5d.reshape(N_te, C, H, W)

# SMELL 2 – Graph-Constant Bottleneck
# The full dataset is moved to GPU as a single massive constant
# tensor up-front. This occupies GPU memory for the entire
# experiment even when only a mini-batch is needed.
X_tr_const = torch.tensor(X_train_back, dtype=torch.float32).to(DEVICE)  # entire dataset on GPU
y_tr_const = torch.tensor(y_train_np,   dtype=torch.long).to(DEVICE)
X_te_const = torch.tensor(X_test_back,  dtype=torch.float32).to(DEVICE)
y_te_const = torch.tensor(y_test_np,    dtype=torch.long).to(DEVICE)

# Still need CPU-side loaders for compatibility
X_tr = torch.tensor(X_train_back, dtype=torch.float32)
y_tr = torch.tensor(y_train_np,   dtype=torch.long)
X_te = torch.tensor(X_test_back,  dtype=torch.float32)
y_te = torch.tensor(y_test_np,    dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# SMELL 1 – Unbounded Graph Expansion
# loss_history retains every loss tensor with its full computation
# graph (grad_fn chain) across all runs. Each tensor keeps the
# entire forward-pass graph alive, so GPU/CPU memory grows
# without bound as runs accumulate.
loss_history = []   # graphs never detached – accumulated across all 10 runs

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f"  Run {run}/10")
    print(f"{'='*60}")

    # SMELL 3 – GPU Released Memory Failure
    # torch.cuda.empty_cache() is deliberately omitted.
    # GPU memory from weights, activations, and optimizer states
    # of previous runs is never explicitly freed; the runtime must
    # rely on Python's garbage collector which does not manage
    # CUDA memory, causing progressive GPU memory leakage.
    # torch.cuda.empty_cache()  <-- intentionally removed

    tracker = EmissionsTracker(
        project_name=f"ResNet18_CIFAR10_Smelly_Run{run}",
        output_dir="./emissions",
        save_to_file=True,
        log_level="warning"
    )
    tracker.start()

    model     = build_resnet18(num_classes=10)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss  = float('inf')
    patience_count = 0
    epoch_stopped  = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss_val = 0.0
        correct, total = 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # SMELL 1 – Unbounded Graph Expansion (continued)
            # Appending the raw loss tensor (with grad_fn) to the
            # persistent list keeps the computation graph alive
            # across iterations and runs.
            loss_history.append(loss)   # graph not detached → memory grows unboundedly

            loss.backward()
            optimizer.step()
            total_loss_val += loss.item() * X_batch.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(y_batch).sum().item()
            total   += X_batch.size(0)

        train_loss = total_loss_val / total
        train_acc  = correct / total

        val_loss, val_acc = eval_epoch(model, test_loader, criterion)
        print(f"  Epoch {epoch:02d}/{EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        epoch_stopped = epoch
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= 2:
                print(f"  Early stopping triggered at epoch {epoch}")
                break

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    _, test_acc = eval_epoch(model, test_loader, criterion)
    prec, rec   = get_precision_recall(model, test_loader)

    run_result = dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_accuracy=test_acc,
        test_precision=prec,
        test_recall=rec,
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    )
    results.append(run_result)
    print(f"Run {run} finished | Epoch stopped: {epoch_stopped} | "
          f"Acc: {test_acc:.4f} | Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg")

    # SMELL 3 – GPU Released Memory Failure (continued)
    # model is NOT deleted; only gc.collect() is called, which
    # targets CPU memory and leaves GPU allocations intact.
    gc.collect()   # CPU-only – GPU weights remain allocated


## Results Summary

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
header = f"{'Run':<5} {'EpochStopped':<14} {'Accuracy':<12} {'Precision':<12} {'Recall':<10} {'Energy(kWh)':<14} {'CO2(kg)':<12}"
print(header)
print("-"*80)
for r in results:
    print(f"{r['run']:<5} {r['epoch_stopped']:<14} {r['test_accuracy']:<12.4f} "
          f"{r['test_precision']:<12.4f} {r['test_recall']:<10.4f} "
          f"{r['energy_kwh']:<14.6f} {r['co2_kg']:<12.6f}")
print("-"*80)
avg_epoch  = np.mean([r['epoch_stopped']    for r in results])
avg_acc    = np.mean([r['test_accuracy']    for r in results])
avg_prec   = np.mean([r['test_precision']   for r in results])
avg_rec    = np.mean([r['test_recall']      for r in results])
avg_energy = np.mean([r['energy_kwh']       for r in results])
avg_co2    = np.mean([r['co2_kg']           for r in results])
print(f"{'AVG':<5} {avg_epoch:<14.1f} {avg_acc:<12.4f} {avg_prec:<12.4f} "
      f"{avg_rec:<10.4f} {avg_energy:<14.6f} {avg_co2:<12.6f}")
print("="*80)
